# 00 · End-to-end workflow (one notebook that drives the .py modules)

Every step below is a function from `src/driftrev/`. The same sequence runs headless via
`python -m driftrev.pipeline --mode update` (cron / Docker scheduler).

| step | function | regime |
|---|---|---|
| 1 download (incremental) | `pipeline.ingest()` | `update` only |
| 2 transform → one table | `pipeline.transform()` / `pipeline.load_snapshot()` | `update` / `file` |
| 3 feature sets + split | `dataset.prepare()` | both |
| 4 train / tune / backtest | `pipeline.train_and_backtest()` | `--retrain` |
| 5 score newest events → signals | `pipeline.publish_live()` | both |
| 6 serve to agents / broker | `service.*`, `broker.preview_orders()` | both |


In [1]:
import sys, warnings, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
plt.rcParams.update({"figure.figsize": (11, 4.5), "axes.grid": True, "grid.alpha": 0.3})
from driftrev import config, store, artifacts, dataset, features, models, simulate, analysis, pipeline
print("data dir:", config.DATA_DIR)

data dir: /Users/mac/Documents/New project/stock-data-analysis/project/data


## 1 · Download (set `FETCH = True` to pull the latest data incrementally)

In [2]:
FETCH = False          # True: Yahoo + FRED + SEC EDGAR + Wikimedia, only the missing days
if FETCH:
    report = pipeline.ingest()
    print(json.dumps(report, indent=1, default=str)[:2000])
with store.connect() as con:
    sizes = {t: con.execute(f"select count(*) from {t}").fetchone()[0]
             for t in ["universe", "prices", "market", "macro", "filings", "eps_facts", "pageviews", "events", "daily_panel"]}
pd.Series(sizes, name="rows").to_frame()

,rows
universe,1678
prices,4837123
market,50226
macro,21565
filings,2021569
eps_facts,370038
pageviews,4544238
events,149055
daily_panel,4835791


## 2 · Transform (or load the snapshot in `file` regime)

In [3]:
TRANSFORM = False      # True: rebuild events + daily panel from the raw tables (~3 min)
data = pipeline.transform() if TRANSFORM else pipeline.load_from_warehouse()
events, panel = data["events"], data["panel"]
print(f"events: {len(events):,}   daily panel: {len(panel):,} rows   tickers: {panel.ticker.nunique():,}")
events[["date", "ticker", "event_family", "ret_1d", "edgar_category", "sue", "growth_future_10d"]].tail()

events: 149,055   daily panel: 4,835,791 rows   tickers: 1,659


,date,ticker,event_family,ret_1d,edgar_category,sue,growth_future_10d
149050,2026-11-13,STE,momentum_breakout,0.042929,offering,0.811537,NaN
149051,2026-11-13,VIR,shock_crash,-0.178218,offering,1.139782,NaN
149052,2026-11-13,VYX,momentum_breakout,0.052209,none,0.072033,NaN
149053,2026-11-13,WMT,momentum_breakout,0.015057,none,-0.061610,NaN
149054,2026-11-13,WWW,momentum_breakout,0.088325,none,0.992157,NaN


## 3 · Feature sets and temporal split

In [4]:
ml, feats = dataset.prepare(events)
print("TO_PREDICT:", dataset.TO_PREDICT)
print(f"NUMERIC: {len(dataset.numeric_features())}  DUMMIES from {dataset.CATEGORICAL}")
print(f"total model features: {len(feats)}")
ml.groupby("split").agg(rows=("ticker", "size"), start=("date", "min"), end=("date", "max"),
                        base_rate=(dataset.TARGET, "mean"))

TO_PREDICT: ['growth_future_5d', 'growth_future_10d', 'growth_future_20d', 'growth_future_5d_vs_spy', 'growth_future_10d_vs_spy', 'growth_future_20d_vs_spy', 'mae_10d', 'mfe_10d', 'is_positive_growth_10d', 'is_positive_growth_10d_vs_spy']
NUMERIC: 152  DUMMIES from ['event_family', 'edgar_category', 'edgar_category_text', 'sector', 'cap_bucket', 'month', 'weekday', 'regime_state', 'bin_reaction', 'bin_rsi', 'bin_sue', 'bin_52w', 'bin_vix', 'bin_attention', 'bin_momentum', 'month_wom']
total model features: 269


,rows,start,end,base_rate
split,,,,
purged,880,2021-12-16,2023-12-29,0.180682
test,43015,2024-01-02,2026-11-13,0.392292
train,79544,2014-01-31,2021-12-15,0.358468
validation,25616,2022-01-03,2023-12-15,0.384916


## 4 · Train + backtest (`RETRAIN=True` refits everything; otherwise reuse the saved model)

In [5]:
RETRAIN = False
if RETRAIN:
    m = pipeline.train_and_backtest(ml, feats, panel, data["market"], tune=False)
metrics = artifacts.load_metrics()
bundle = models.load_bundle()
print("production model:", bundle["name"], "threshold:", bundle["threshold"],
      "val AUC %.3f  test AUC %.3f" % (bundle["val_auc"], bundle["test_auc"]))
pd.DataFrame(metrics["strategies"] + metrics["benchmarks"])[["name", "cagr", "sharpe", "max_drawdown", "n_trades"]]

production model: ensemble threshold: 0.52 val AUC 0.557  test AUC 0.546


,name,cagr,sharpe,max_drawdown,n_trades
0,all_events_unconditional,-0.084795,-0.084161,-0.578528,360.0
1,model_top5,0.083536,0.431419,-0.180397,102.0
2,model_top3_concentrated,-0.013660,0.139793,-0.319849,81.0
3,model_top5_risk_managed,0.013679,0.167295,-0.208951,107.0
4,model_pead_only,0.035789,0.305014,-0.158863,56.0
5,model_crash_only,0.163854,0.704908,-0.191891,66.0
6,model_breakout_only,0.042698,0.474534,-0.126726,26.0
7,model_long_short,0.048866,0.412897,-0.155331,296.0
8,guide_armA_momentum_breakout,-0.005072,-0.030043,-0.117098,568.0
9,guide_armB_plus_consolidation,0.003057,0.097190,-0.050396,146.0


## 5 · Score the newest events → today's new trades

In [6]:
live = pipeline.publish_live(ml, panel, data["universe"], bundle)
print(live)
sig = artifacts.load_frame("signals")
recent = artifacts.load_frame("events")
cols = ["date", "ticker", "company", "event_family", "ret_1d", "prob", "tier", "expected_net_return",
        "entry_ref_price", "stop_loss_price", "take_profit_price", "exit_by_date", "edgar_category"]
print("latest session:", live["data_through"])
(sig if len(sig) else recent.head(15))[cols]

{'data_through': '2026-11-13', 'n_signals_today': 20, 'n_actionable_today': 0, 'n_events_90d': 3929}
latest session: 2026-11-13


,date,ticker,company,event_family,ret_1d,prob,tier,expected_net_return,entry_ref_price,stop_loss_price,take_profit_price,exit_by_date,edgar_category
0,2026-11-13,AMAT,Applied Materials,earnings_down,-0.027431,0.485071,WATCH,0.011969,370.500000,314.92,463.12,2026-11-30,earnings
1,2026-11-13,FUN,Six Flags,momentum_breakout,0.032172,0.473478,SKIP,0.008569,11.550000,9.97,16.29,2026-11-26,none
2,2026-11-13,FLO,Flowers Foods,earnings_down,-0.079867,0.450158,SKIP,0.008569,5.530000,4.70,6.91,2026-11-30,earnings
3,2026-11-13,CSCO,Cisco,earnings_down,-0.062445,0.441088,SKIP,0.011969,95.639999,81.29,119.55,2026-11-30,earnings
4,2026-11-13,ARWR,Arrowhead Pharmaceuticals,earnings_up,0.013362,0.439019,SKIP,0.003408,47.779999,40.61,59.72,2026-11-30,earnings
5,2026-11-13,DIOD,Diodes Incorporated,earnings_down,-0.047489,0.427458,SKIP,0.001408,87.250000,74.16,109.06,2026-11-30,earnings
6,2026-11-13,IBP,"Installed Building Products, Inc.",momentum_breakout,0.023418,0.420114,SKIP,0.001408,229.000000,208.04,291.87,2026-11-26,none
7,2026-11-13,CXM,"Sprinklr, Inc.",momentum_breakout,0.022556,0.419238,SKIP,0.001408,6.800000,6.35,8.14,2026-11-26,none
8,2026-11-13,PHIN,"PHINIA, Inc.",momentum_breakout,0.011860,0.415874,SKIP,0.001408,73.370003,68.56,87.80,2026-11-26,none
9,2026-11-13,VYX,NCR Voyix,momentum_breakout,0.052209,0.411004,SKIP,-0.000155,10.480000,9.35,13.86,2026-11-26,none


## 6 · What a 24/7 agent sees (same functions as the REST API / MCP server)

In [7]:
from driftrev import service, broker
print(service.latest_signals(limit=5)["summary"])
print(json.dumps(service.regime(), indent=1, default=str)[:800])
preview = broker.preview_orders(max_positions=5, capital=10_000)
pd.DataFrame(preview.get("orders", []))

Data through 2026-11-13; market regime: normal. 5 signal(s). Top: AMAT (earnings_down) tier WATCH, P(+2% in 10d)=49%, weight 0%, stop 314.92; FUN (momentum_breakout) tier SKIP, P(+2% in 10d)=47%, weight 0%, stop 9.97; FLO (earnings_down) tier SKIP, P(+2% in 10d)=45%, weight 0%, stop 4.70; CSCO (earnings_down) tier SKIP, P(+2% in 10d)=44%, weight 0%, stop 81.29; ARWR (earnings_up) tier SKIP, P(+2% in 10d)=44%, weight 0%, stop 40.61. Research output, not investment advice.
{
 "data_through": "2026-11-13",
 "regime": {
  "state": "normal",
  "points": 0,
  "as_of": "2026-11-13",
  "flags": {
   "flag_trend": 0,
   "flag_vix_term": 0,
   "flag_corr": 0,
   "flag_breadth": 0,
   "flag_credit": 0
  },
  "vix": 19.299999237060547,
  "vix_term": 0.9359844059918124,
  "breadth_above_200": 0.5569023569023569,
  "explain": "1 point per trigger: SPY<200d MA, VIX/VIX3M>1, sector-correlation pctile>0.9, breadth<40% above 200d MA, Baa-10y spread +25bp/20d. 0-1 normal, 2-3 elevated, 4+ stressed."
 },


""
